# EPL sigma_los 对比

比较本地 `jampy` 与 `jaxpy` 在零转动假设下的 EPL 质量模型 `sigma_los`（即 `Vrms`）分布。

In [ ]:
import os
import sys
import importlib
import pickle as pkl
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cpu"

import matplotlib
matplotlib.use("Agg")

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

plt.ioff()

root = Path("/mnt/d/lensing/jaxpy/jaxpy/jaxpy")
png_dir = root / "png"
png_dir.mkdir(exist_ok=True)
sys.path.insert(0, str(root))
sys.path.insert(0, "/mnt/d/lensing/Jampy/jampy-8.1.4")

import param_util
from SLCOSMO import tool
from MGE_jax import MGE
from jam_axi_proj_jax import jam_axi_proj
import jampy.axi.jam_axi_proj as jam_base_mod

jam_base_mod = importlib.reload(jam_base_mod)
from jampy.axi.jam_axi_proj import jam_axi_proj as jam_base

theta_E = 1.397
gamma = 2.027
q_mass = 0.946
z_lens = 0.222
z_source = 0.609
fwhm_psf = 0.6

cosmology = {
    "Omegam": 0.3,
    "Omegak": 0.0,
    "w0": -1.0,
    "wa": 0.0,
    "h0": 70.0,
}

lens_light = pkl.load(open(root / "lens_light_jackpot.pkl", "rb"))[0]
surf_lum = jnp.asarray(lens_light["amp"])
sigma_lum = jnp.asarray(lens_light["sigma"])
_, qobs_lum = param_util.ellipticity2phi_q(lens_light["e1"], lens_light["e2"])
qobs_lum = jnp.asarray(qobs_lum)
beta = jnp.zeros_like(surf_lum)

grid_size = 60
x_lin = np.linspace(-2.0, 2.0, grid_size)
y_lin = np.linspace(-2.0, 2.0, grid_size)
x_mesh, y_mesh = np.meshgrid(x_lin, y_lin, indexing="xy")
xbin = x_mesh.ravel()
ybin = y_mesh.ravel()
goodbins = np.ones_like(xbin, dtype=bool)
pixsize = float(x_lin[1] - x_lin[0])
sigmapsf = jnp.array([fwhm_psf / 2.355])
normpsf = jnp.array([1.0])

distance = float(np.asarray(tool.dldsdls(z_lens, z_source, cosmology, n=20)[0]).ravel()[0])
mge_epl = MGE(
    tool.EPL_msunmpc,
    "thetaE",
    n_gauss=10,
    n_terms=28,
    sigma_start_mult=1 / 100,
    sigma_end_mult=50,
)
surf_pot, sigma_pot = mge_epl.decompose(
    thetaE=theta_E,
    gamma=gamma,
    zl=z_lens,
    zs=z_source,
    cosmology=cosmology,
)
qobs_pot = jnp.full_like(surf_pot, q_mass)

def derive_grid_params(x, y, sigma_lum, qobs_lum, sigma_psf, pixsize, nrad=20, nang=10):
    qmed = np.median(np.asarray(qobs_lum))
    rell = np.hypot(np.asarray(x), np.asarray(y) / qmed)
    step = float(np.min(np.asarray(sigma_psf)) / 4)
    mx = float(3 * np.max(np.asarray(sigma_psf)) + pixsize / np.sqrt(2))
    rmax = float(np.max(rell) + mx)
    nx = int(np.ceil(rmax / step))
    ny = int(np.ceil(rmax * qmed / step))
    nk = int(np.ceil(mx / step))
    return nx, ny, nk, step

def to_base_kwargs(kwargs):
    scalar_keys = {"inc", "mbh", "distance", "pixsize", "step"}
    pass_keys = {"logistic", "moment", "align", "ml"}
    out = {}
    for key, value in kwargs.items():
        if value is None:
            out[key] = value
        elif key in scalar_keys:
            out[key] = float(value)
        elif key in pass_keys:
            out[key] = value
        elif key == "goodbins":
            out[key] = np.asarray(value, dtype=bool)
        else:
            out[key] = np.asarray(value)
    return out

nx, ny, nk, step = derive_grid_params(xbin, ybin, sigma_lum, qobs_lum, sigmapsf, pixsize)

common_kwargs = dict(
    surf_lum=surf_lum,
    sigma_lum=sigma_lum,
    qobs_lum=qobs_lum,
    surf_pot=surf_pot,
    sigma_pot=sigma_pot,
    qobs_pot=qobs_pot,
    inc=90.0,
    mbh=0.0,
    distance=distance,
    xbin=xbin,
    ybin=ybin,
    sigmapsf=sigmapsf,
    normpsf=normpsf,
    beta=beta,
    pixsize=pixsize,
    logistic=False,
    moment="zz",
    goodbins=goodbins,
    align="sph",
    ml=None,
    nrad=20,
    nang=10,
    nlos=1500,
    step=step,
)

sigma_base = np.asarray(jam_base(**to_base_kwargs(common_kwargs), plot=False, quiet=True).model)
sigma_jax, *_ = jam_axi_proj().get_kinematics(**dict(common_kwargs, nx=nx, ny=ny, nk=nk), quiet=True)
sigma_jax = np.asarray(sigma_jax)

def to_grid(values):
    return np.asarray(values).reshape(grid_size, grid_size)

base_grid = to_grid(sigma_base)
jax_grid = to_grid(sigma_jax)
pct_grid = 100.0 * (jax_grid - base_grid) / np.where(np.abs(base_grid) > np.finfo(float).eps, np.abs(base_grid), np.nan)
abs_diff = np.abs(sigma_jax - sigma_base)

vmin = np.nanmin([base_grid.min(), jax_grid.min()])
vmax = np.nanmax([base_grid.max(), jax_grid.max()])
dmax_pct = np.nanmax(np.abs(pct_grid))

fig = plt.figure(figsize=(13.4, 5.2))
gs = fig.add_gridspec(1, 5, width_ratios=[1.0, 1.0, 0.055, 1.0, 0.055], wspace=0.06)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 3])]
axes[1].sharex(axes[0])
axes[1].sharey(axes[0])
axes[2].sharex(axes[0])
axes[2].sharey(axes[0])
cax_main = fig.add_subplot(gs[0, 2])
cax_diff = fig.add_subplot(gs[0, 4])
fig.subplots_adjust(left=0.065, right=0.955, bottom=0.14, top=0.82)
diff_norm = TwoSlopeNorm(vmin=-dmax_pct, vcenter=0.0, vmax=dmax_pct) if dmax_pct > 0 else None

pc0 = axes[0].pcolormesh(x_lin, y_lin, base_grid, shading="auto", cmap="inferno", vmin=vmin, vmax=vmax)
pc1 = axes[1].pcolormesh(x_lin, y_lin, jax_grid, shading="auto", cmap="inferno", vmin=vmin, vmax=vmax)
pc2 = axes[2].pcolormesh(x_lin, y_lin, np.nan_to_num(pct_grid, nan=0.0), shading="auto", cmap="RdBu_r", norm=diff_norm)

contour_levels = np.linspace(vmin, vmax, 7)
axes[0].contour(x_lin, y_lin, base_grid, levels=contour_levels, colors="white", linewidths=0.55, alpha=0.35)
axes[1].contour(x_lin, y_lin, jax_grid, levels=contour_levels, colors="white", linewidths=0.55, alpha=0.35)

panel_titles = ["Base", "JAX", "Residual"]
panel_notes = [
    r"$\sigma_{\rm los}$ from jampy",
    r"$\sigma_{\rm los}$ from jaxpy",
    r"$(\mathrm{JAX}-\mathrm{Base})\,[\%]$",
]

for i, (ax, title, note) in enumerate(zip(axes, panel_titles, panel_notes)):
    ax.set_title(title, fontsize=13, pad=10, fontweight="semibold")
    ax.set_xlim(x_lin.min(), x_lin.max())
    ax.set_ylim(y_lin.min(), y_lin.max())
    ax.set_xticks(np.linspace(-2, 2, 5))
    ax.set_yticks(np.linspace(-2, 2, 5))
    ax.tick_params(labelsize=10)
    if i > 0:
        ax.tick_params(labelleft=False)
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1)
    ax.text(
        0.03,
        0.96,
        note,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9.5,
        color="white" if i < 2 else "black",
        bbox=dict(
            boxstyle="round,pad=0.26",
            facecolor=(0, 0, 0, 0.20) if i < 2 else (1, 1, 1, 0.78),
            edgecolor="none",
        ),
    )

fig.supxlabel("x [arcsec]", fontsize=12, y=0.06)
fig.supylabel("y [arcsec]", fontsize=12, x=0.03)

cbar_main = fig.colorbar(pc1, cax=cax_main, orientation="vertical")
cbar_main.set_label(r"$\sigma_{\rm los}$ [km s$^{-1}$]", fontsize=11)
cbar_main.ax.tick_params(labelsize=9)
cbar_diff = fig.colorbar(pc2, cax=cax_diff, orientation="vertical")
cbar_diff.set_label(r"Residual [\%]", fontsize=11)
cbar_diff.ax.tick_params(labelsize=9)

fig.suptitle("Zero-rotation EPL $\sigma_{los}$ comparison ($\mathrm{sph}$)", fontsize=16, fontweight="semibold", y=0.96)
fig.text(
    0.5,
    0.905,
    f"$\\theta_E={theta_E:.3f}$, $\\gamma={gamma:.3f}$, $q={q_mass:.3f}$, $z_l={z_lens:.3f}$, $z_s={z_source:.3f}$, PSF FWHM $={fwhm_psf:.1f}$ arcsec",
    ha="center",
    va="center",
    fontsize=11,
    color="#444444",
)

out_png = png_dir / "epl_sigma_compare_thetaE1.397_gamma2.027_q0.946_zl0.222_zs0.609_fwhm0.6_sph_styled.png"
fig.savefig(out_png, dpi=220, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)

finite_pct = pct_grid[np.isfinite(pct_grid)]
print(f"saved figure: {out_png.name}")
print("zero rotation assumed -> sigma_los = Vrms")
print(f"theta_E={theta_E}, gamma={gamma}, q={q_mass}, zl={z_lens}, zs={z_source}, FWHM={fwhm_psf}")
print(f"|Δsigma_los| mean={abs_diff.mean():.6g} km/s, median={np.median(abs_diff):.6g} km/s, max={abs_diff.max():.6g} km/s")
print(f"max |Δ%|={np.max(np.abs(finite_pct)):.6g}%")

In [ ]:
from time import perf_counter

n_repeat = 5
base_kwargs_bench = to_base_kwargs(common_kwargs)
jax_kwargs_bench = dict(common_kwargs, nx=nx, ny=ny, nk=nk)

jax.clear_caches()
jax_obj_bench = jam_axi_proj()

t0 = perf_counter()
sigma_jax_warm, *_ = jax_obj_bench.get_kinematics(**jax_kwargs_bench, quiet=True)
jax.block_until_ready(sigma_jax_warm)
jax_first_call = perf_counter() - t0

sigma_jax_settle, *_ = jax_obj_bench.get_kinematics(**jax_kwargs_bench, quiet=True)
jax.block_until_ready(sigma_jax_settle)

base_times = []
for _ in range(n_repeat):
    t0 = perf_counter()
    sigma_base_bench = jam_base(**base_kwargs_bench, plot=False, quiet=True).model
    base_times.append(perf_counter() - t0)

jax_times = []
for _ in range(n_repeat):
    t0 = perf_counter()
    sigma_jax_bench, *_ = jax_obj_bench.get_kinematics(**jax_kwargs_bench, quiet=True)
    jax.block_until_ready(sigma_jax_bench)
    jax_times.append(perf_counter() - t0)

base_times = np.asarray(base_times)
jax_times = np.asarray(jax_times)
speedup_median = np.median(base_times) / np.median(jax_times)
speedup_mean = np.mean(base_times) / np.mean(jax_times)

print("benchmark setup: same EPL sigma_los call, JAX first-call compile excluded from steady-state timing")
print(f"jax first call (compile + execute): {jax_first_call:.6f} s")
print(f"jampy times [s]: {np.array2string(base_times, precision=6, separator=', ')}")
print(f"jaxpy times [s]: {np.array2string(jax_times, precision=6, separator=', ')}")
print(f"jampy median: {np.median(base_times):.6f} s")
print(f"jaxpy median: {np.median(jax_times):.6f} s")
print(f"jampy mean: {np.mean(base_times):.6f} s")
print(f"jaxpy mean: {np.mean(jax_times):.6f} s")
print(f"steady-state speedup (median): {speedup_median:.3f}x")
print(f"steady-state speedup (mean): {speedup_mean:.3f}x")
